# TP L21 — DQN pour un choix de relais

**Public :** première pratique du deep RL · **Durée :** 75 à 90 min

Nous entraînons un DQN minimal dans un environnement réseau **synthétique**. La progression est inspirée du notebook DQN de [FIDLE](https://gricad-gitlab.univ-grenoble-alpes.fr/talks/fidle/-/blob/master/DRL.PyTorch/FIDLE_DQNfromScratch.ipynb), mais l'implémentation ci-dessous est originale et utilise l'API Gymnasium actuelle.

> Ce TP illustre un mécanisme. Il ne constitue ni une validation terrain, ni une preuve de sûreté, ni une politique déployable.

## Objectifs

- lire le contrat `reset` / `step` d'un environnement ;
- comprendre le **rejeu d'expérience** et le **réseau cible** ;
- comparer DQN à une politique aléatoire et à une heuristique ;
- évaluer plusieurs KPI et traiter les contraintes hors de la récompense.

In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib.pyplot as plt
import numpy as np

LAB_ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "dqn_relay_lab.py").is_file()
)
sys.path.insert(0, str(LAB_ROOT))

from dqn_relay_lab import (
    DQNConfig, QNetwork, RelayQueueEnv, _optimize, always_relay_a_policy,
    dqn_policy, evaluate_policy, filter_action, projected_queue_policy,
    random_policy, train_dqn,
)

## 1. Le contrat de l'environnement

L'observation normalisée contient les deux files, l'arrivée, les services et la disponibilité des relais. Tous les contrôleurs reçoivent exactement cette observation. Le budget de pertes restant est caché : le problème est partiellement observable (POMDP) et le DQN sans mémoire est une approximation. Une même observation et une même action peuvent terminer ou continuer selon les pertes accumulées dans l'historique. `terminated` signale l'épuisement de ce budget ; `truncated` signale une limite externe de collecte, après laquelle la tâche pourrait continuer. Un horizon fini intrinsèque demanderait le temps restant dans l'état et une terminaison à échéance.

In [ ]:
env = RelayQueueEnv(horizon=8)
observation, info_reset = env.reset(
    seed=3, options={'phase_offset': 1, 'queue_a': 0, 'queue_b': 0}
)
action_appliquee, bloquee = filter_action(0, observation)
transition = env.step(action_appliquee)
obs_suivante, recompense, terminated, truncated, info = transition
print("observation initiale :", observation)
print("proposition 0 → action", action_appliquee, "bloquée=", bloquee)
print("step →", recompense, terminated, truncated, info)
assert env.observation_space.contains(observation)
assert env.action_space.contains(action_appliquee)
assert bloquee and action_appliquee == 1

## 2. Ce que le DQN apprend

Le réseau estime une valeur Q par action. Le rejeu casse une partie des corrélations temporelles en échantillonnant d'anciennes transitions. Le réseau cible stabilise la cible TD. Pendant une troncature, le bootstrap est conservé ; il s'arrête uniquement lors d'une vraie terminaison.

In [ ]:
print(inspect.getsource(QNetwork))
print(inspect.getsource(_optimize))

## 3. Deux références avant l'apprentissage

Une expérience RL sans référence ne permet pas de savoir si la complexité ajoutée sert à quelque chose. La politique aléatoire et l'heuristique modèle-informée reçoivent ici le même vecteur observable que le DQN ; aucune ne lit l'état privé de l'environnement.

In [ ]:
seeds_evaluation = range(1000, 1040)
mesures = {
    "aléatoire": evaluate_policy(random_policy, seeds=seeds_evaluation),
    "heuristique": evaluate_policy(projected_queue_policy, seeds=seeds_evaluation),
}
for nom, valeurs in mesures.items():
    print(nom, valeurs)

## 4. Entraîner le DQN

Une graine fixe rend cette exécution rejouable, sans mesurer à elle seule l'incertitude d'entraînement. La preuve retenue lance cinq graines indépendantes ; ici, la courbe reste un diagnostic et la comparaison finale porte sur des épisodes séparés.

In [ ]:
configuration = DQNConfig(seed=7, episodes=180)
entrainement = train_dqn(configuration)
print(f"{entrainement.steps} transitions, ε final={entrainement.final_epsilon:.3f}")

fenetre = 20
moyenne_mobile = np.convolve(
    entrainement.episode_returns, np.ones(fenetre) / fenetre, mode="valid"
)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(range(fenetre, len(entrainement.episode_returns) + 1), moyenne_mobile)
ax.set(xlabel="épisode", ylabel="retour moyen (20 épisodes)", title="Apprentissage DQN")
ax.grid(alpha=0.25)
plt.show()

## 5. Comparaison tenue à l'écart

On réutilise exactement les mêmes seeds et le même horizon pour les trois contrôleurs. Le retour agrège l'objectif choisi ; les paquets livrés et perdus évitent de masquer un compromis opérationnel.

In [ ]:
mesures["DQN"] = evaluate_policy(
    dqn_policy(entrainement.model), seeds=seeds_evaluation
)
for nom, valeurs in mesures.items():
    print(
        f"{nom:11s} retour={valeurs['mean_return']:7.2f}  "
        f"livrés={valeurs['mean_delivered']:5.1f}  perdus={valeurs['mean_dropped']:5.1f}  "
        f"propositions bloquées={valeurs['blocked_proposal_rate']:.1%}"
    )

## 6. Prouver la frontière de sûreté

La politique de test suivante propose toujours le relais A, même lorsqu'il est indisponible. Le filtre doit bloquer certaines propositions et n'appliquer aucune action interdite. Cette propriété ne dépend pas de la récompense.

In [ ]:
preuve_filtre = evaluate_policy(
    always_relay_a_policy, seeds=range(1000, 1040),
    drop_termination_threshold=None,
)
assert preuve_filtre['mean_blocked_proposals'] > 0
assert preuve_filtre['applied_constraint_violations'] == 0
print(preuve_filtre)

## 7. Lire le résultat sans surpromettre

Le DQN doit dépasser l'aléatoire sur ce banc synthétique. L'heuristique peut pourtant rester meilleure sur certains KPI : un meilleur retour ne signifie pas une domination universelle. Les intervalles multi-graines et les résultats négatifs sont conservés dans `results/benchmark_results.json`.

## 8. Extensions guidées

1. Faites varier `gamma` : quel KPI est le plus sensible ?
2. Doublez `target_sync_steps` : observez la stabilité, pas seulement le score final.
3. Modifiez le poids des pertes dans la récompense : quel comportement opportuniste apparaît ?
4. Changez le calendrier d'indisponibilité. Le filtre reste-t-il inviolable ?
5. Rédigez une conclusion en quatre lignes : hypothèse, résultat, limite, expérience suivante.